# Notebook 5: Machine Learning Model Building

**Project:** End-to-End Retail Sales Analytics and Forecasting using Python, Machine Learning, and Power BI

This notebook trains and compares multiple regression models to predict **Sales**.


In [1]:
import pandas as pd
import numpy as np
import joblib

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder

from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

try:
    from xgboost import XGBRegressor
    xgb_available = True
except ImportError:
    xgb_available = False
    print("XGBoost is not installed. Run: pip install xgboost")

df = pd.read_csv("feature_engineered_retail_sales.csv")
df.head()

,Row_ID,Order_ID,Order_Date,Ship_Date,Ship_Mode,Customer_ID,Customer_Name,Segment,Country,City,...,Weekday,Is_Weekend,Sales_per_Quantity,Profit_Margin,Order_Week,Day_of_Week,Shipping_Days,Discount_Flag,Customer_Order_Count,Product_Order_Count
0,1,CA-2016-152156,2016-11-08,2016-11-11,2,CG-12520,Claire Gute,0,United States,194,...,Tuesday,0,130.9800,16.00,45,Tuesday,3,0,5,4
1,2,CA-2016-152156,2016-11-08,2016-11-11,2,CG-12520,Claire Gute,0,United States,194,...,Tuesday,0,243.9800,30.00,45,Tuesday,3,0,5,12
2,3,CA-2016-138688,2016-06-12,2016-06-16,2,DV-13045,Darrin Van Huff,1,United States,266,...,Sunday,1,7.3100,47.00,23,Sunday,4,0,9,7
3,4,US-2015-108966,2015-10-11,2015-10-18,3,SO-20335,Sean O'Donnell,0,United States,153,...,Sunday,1,191.5155,-40.00,41,Sunday,7,1,15,8
4,5,US-2015-108966,2015-10-11,2015-10-18,3,SO-20335,Sean O'Donnell,0,United States,153,...,Sunday,1,11.1840,11.25,41,Sunday,7,1,15,5


In [2]:
print(df.shape)
df.info()

(9994, 36)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9994 entries, 0 to 9993
Data columns (total 36 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   Row_ID                9994 non-null   int64  
 1   Order_ID              9994 non-null   object 
 2   Order_Date            9994 non-null   object 
 3   Ship_Date             9994 non-null   object 
 4   Ship_Mode             9994 non-null   int64  
 5   Customer_ID           9994 non-null   object 
 6   Customer_Name         9994 non-null   object 
 7   Segment               9994 non-null   int64  
 8   Country               9994 non-null   object 
 9   City                  9994 non-null   int64  
 10  State                 9994 non-null   int64  
 11  Postal_Code           9994 non-null   int64  
 12  Region                9994 non-null   int64  
 13  Product_ID            9994 non-null   object 
 14  Category              9994 non-null   int64  
 15  Sub-Catego

In [3]:
target = "Sales"

drop_cols = [
    target,
    "Order_ID",
    "Customer_Name",
    "Product_Name"
]

X = df.drop(columns=[c for c in drop_cols if c in df.columns])
y = df[target]

categorical_cols = X.select_dtypes(include=["object"]).columns.tolist()
numeric_cols = X.select_dtypes(exclude=["object"]).columns.tolist()

preprocessor = ColumnTransformer([
    ("num", Pipeline([
        ("imputer", SimpleImputer(strategy="median"))
    ]), numeric_cols),
    ("cat", Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore"))
    ]), categorical_cols)
])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(X_train.shape, X_test.shape)

(7995, 32) (1999, 32)


In [4]:
models = {
    "Linear Regression": LinearRegression(),
    "Decision Tree": DecisionTreeRegressor(random_state=42),
    "Random Forest": RandomForestRegressor(n_estimators=100, random_state=42),
    "Gradient Boosting": GradientBoostingRegressor(random_state=42)
}

if xgb_available:
    models["XGBoost"] = XGBRegressor(
        random_state=42,
        n_estimators=200,
        learning_rate=0.1,
        max_depth=6,
        objective="reg:squarederror"
    )

results = []
best_model = None
best_r2 = -999
best_pipeline = None

for name, model in models.items():
    pipe = Pipeline([
        ("preprocessor", preprocessor),
        ("model", model)
    ])
    pipe.fit(X_train, y_train)
    preds = pipe.predict(X_test)

    mae = mean_absolute_error(y_test, preds)
    rmse = np.sqrt(mean_squared_error(y_test, preds))
    r2 = r2_score(y_test, preds)

    results.append([name, mae, rmse, r2])

    if r2 > best_r2:
        best_r2 = r2
        best_model = name
        best_pipeline = pipe

results_df = pd.DataFrame(
    results,
    columns=["Model","MAE","RMSE","R2 Score"]
).sort_values("R2 Score", ascending=False)

results_df

,Model,MAE,RMSE,R2 Score
0,Linear Regression,116.886314,355.935508,0.785525
1,Decision Tree,25.182573,362.338405,0.777739
3,Gradient Boosting,31.430720,369.539536,0.768817
2,Random Forest,22.370086,382.625577,0.752154
4,XGBoost,26.588858,384.340062,0.749928


In [5]:
print("Best Model:", best_model)
joblib.dump(best_pipeline, "sales_model.pkl")
print("Model saved as sales_model.pkl")

Best Model: Linear Regression
Model saved as sales_model.pkl


In [6]:
predictions = best_pipeline.predict(X_test)

comparison = pd.DataFrame({
    "Actual Sales": y_test.values,
    "Predicted Sales": predictions
})

comparison["Difference"] = comparison["Actual Sales"] - comparison["Predicted Sales"]

comparison.head()

,Actual Sales,Predicted Sales,Difference
0,563.808,493.353086,70.454914
1,36.672,-26.426422,63.098422
2,37.300,-22.157944,59.457944
3,212.058,218.568535,-6.510535
4,171.288,165.883026,5.404974


In [7]:
comparison.to_csv("sales_predictions.csv", index=False)
results_df.to_csv("model_comparison_results.csv", index=False)

print("Prediction file and model comparison report saved successfully.")

Prediction file and model comparison report saved successfully.
